# Neural Network for Weekly Sales Forecasting

Trong notebook này, chúng ta sẽ:
- Chuẩn hoá dữ liệu đầu vào.
- Xây dựng mô hình mạng nơ-ron nhân tạo (Neural Network) để dự đoán `Weekly_Sales`.
- Đánh giá mô hình bằng các chỉ số Mean Squared Error (MSE) và R-squared (R²).

In [ ]:
# Import thư viện cần thiết
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Đọc dữ liệu (giả định file trong cùng thư mục, tên là 'train.csv' và 'features.csv')
train_df = pd.read_csv('train.csv')
features_df = pd.read_csv('features.csv')

# Kết hợp dữ liệu dựa trên Store và Date
merged_df = pd.merge(train_df, features_df, on=['Store', 'Date'], how='inner')
print("Kích thước dữ liệu sau khi kết hợp:", merged_df.shape)

### 1. Tiền xử lý dữ liệu

In [ ]:
# Trích xuất các đặc trưng từ cột Date
merged_df['Date'] = pd.to_datetime(merged_df['Date'])
merged_df['Year'] = merged_df['Date'].dt.year
merged_df['Month'] = merged_df['Date'].dt.month
merged_df['Week'] = merged_df['Date'].dt.isocalendar().week
merged_df['DayOfWeek'] = merged_df['Date'].dt.dayofweek

# Chọn các cột cần thiết
features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Week', 'DayOfWeek']
X = merged_df[features]
y = merged_df['Weekly_Sales']

# Kiểm tra giá trị bị thiếu
print("Giá trị bị thiếu:")
print(X.isnull().sum())

# Điền khuyết nếu cần thiết
X.fillna(method='ffill', inplace=True)

# Chuẩn hoá dữ liệu bằng Min-Max Scaling
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Chia dữ liệu thành tập huấn luyện và kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print("Kích thước tập huấn luyện:", X_train.shape, y_train.shape)

### 2. Xây dựng mô hình mạng nơ-ron nhân tạo

In [ ]:
# Xây dựng mô hình Neural Network
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer
])

# Biên dịch mô hình
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Hiển thị kiến trúc mô hình
model.summary()

### 3. Huấn luyện mô hình

In [ ]:
# Huấn luyện mô hình
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=1
)

### 4. Đánh giá mô hình

In [ ]:
# Dự đoán trên tập kiểm tra
y_pred = model.predict(X_test)

# Tính toán MSE và R2
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared (R²): {r2}")

### 5. Vẽ biểu đồ tổn thất (Loss) trong quá trình huấn luyện

In [ ]:
# Vẽ biểu đồ loss
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss during Training and Validation')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

### Tổng kết:
- Mô hình mạng nơ-ron đã được xây dựng và huấn luyện trên dữ liệu chuẩn hoá.
- Độ chính xác của mô hình được đánh giá qua MSE và R².
- Biểu đồ tổn thất giúp kiểm tra quá trình hội tụ của mô hình.